# Aplicación RAG con NVIDIA NIM en Google Colab

Esta versión es **autosuficiente**: configura NVIDIA NIM, carga los documentos, crea un índice Qdrant temporal y publica una interfaz Gradio.

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3-super-120b-a12b`

> En Colab, ejecuta las celdas en orden. La última celda mostrará una URL pública temporal de Gradio.


## 1. Instalar dependencias

Después de la instalación no suele ser necesario reiniciar el runtime. Si Colab lo solicita, reinícialo y vuelve a ejecutar desde la celda siguiente.


In [1]:
%pip install -qU gradio langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 15.4 MB/s eta 0:00:00


## 2. Configurar la API key

En Colab, abre **🔑 Secrets**, crea un secreto llamado `NVIDIA_API_KEY` y habilita el acceso para este notebook. También puedes introducirla de forma oculta cuando la celda la solicite.


In [3]:
import os
import json
import getpass
from pathlib import Path
from typing import List, Dict

import gradio as gr
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3.5-super-vl-120b-a12b"
EMBEDDING_MODEL = "nvidia/llama-nemotron-embed-vl-1b-v2"
COLLECTION = "company_docs_gradio"

print("✅ Configuración NVIDIA NIM lista")


✅ Configuración NVIDIA NIM lista


## 3. Cargar los documentos

Si quieres usar documentos propios, súbelos a `data/company_documents.json` con los campos `id`, `title`, `content` y, opcionalmente, `metadata`. Si el archivo no existe, la aplicación usa un conjunto de demostración.


In [4]:
from pathlib import Path
import json
from typing import List, Dict

def load_documents(path: Path) -> List[Dict]:
    if not path.exists():
        raise FileNotFoundError(
            f"No se encontró el archivo: {path.resolve()}. "
            "Sube company_documents.json a la carpeta data o a la raíz de Colab."
        )

    with path.open(encoding="utf-8") as file:
        documents = json.load(file)

    required = {"id", "title", "content"}
    for index, document in enumerate(documents):
        missing = required - document.keys()
        if missing:
            raise ValueError(
                f"Al documento {index} le faltan campos: {missing}"
            )
        document.setdefault("metadata", {})

    return documents


# Preferir data/company_documents.json; aceptar también el archivo en la raíz.
documents_path = Path("data/company_documents.json")
if not documents_path.exists():
    documents_path = Path("company_documents.json")

company_documents = load_documents(documents_path)

print(f"✅ Documentos cargados: {len(company_documents)}")
print(f"📂 Ruta utilizada: {documents_path}")
for doc in company_documents:
    print(f"- {doc['title']}")

✅ Documentos cargados: 8
📂 Ruta utilizada: company_documents.json
- Política de trabajo remoto
- Política de vacaciones
- Política de reembolso de gastos
- Política de descuentos comerciales
- Horario de atención al cliente
- Proceso de selección de personal
- Programa de capacitación
- Política de soporte técnico


## 4. Construir el pipeline RAG

Esta celda crea los fragmentos, calcula embeddings con NVIDIA NIM y los indexa en Qdrant. El índice vive en memoria mientras la sesión de Colab esté activa.


In [5]:
def build_pipeline(documents: List[Dict]):
    embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")
    llm = ChatNVIDIA(
        model=GENERATION_MODEL,
        temperature=0.2,
        top_p=0.95,
        max_completion_tokens=700,
    )

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=40,
        separators=["\n\n", "\n", ".", " "],
    )

    chunks: List[Document] = []
    for document in documents:
        for index, text in enumerate(splitter.split_text(document["content"])):
            chunks.append(Document(
                page_content=text,
                metadata={
                    "source_id": document["id"],
                    "title": document["title"],
                    "chunk_index": index,
                    **document.get("metadata", {}),
                },
            ))

    # La dimensión se obtiene del modelo y no queda codificada manualmente.
    embedding_dim = len(embeddings.embed_query("prueba de dimensión"))
    qdrant_client = QdrantClient(":memory:")
    qdrant_client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )

    vectorstore = QdrantVectorStore(
        client=qdrant_client,
        collection_name=COLLECTION,
        embedding=embeddings,
    )
    vectorstore.add_documents(chunks)

    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.3},
    )

    # Prompt modificado para la tarea A3
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """Eres un asistente empresarial que responde preguntas
sobre NovaTech utilizando exclusivamente los documentos proporcionados.

Sigue estrictamente estas instrucciones:
1. Responde siempre en español.
2. Utiliza únicamente la información disponible en el contexto.
   No inventes datos ni utilices conocimientos externos.
3. Si el contexto no contiene información suficiente para responder,
   responde exactamente:
   No encuentro esa información en los documentos.
   No añadas ninguna explicación adicional en ese caso.
4. Cuando respondas utilizando información de un documento,
   incluye el título de la fuente entre corchetes.
   Por ejemplo: [Política de trabajo remoto].
5. Responde con un máximo de tres oraciones.
6. Sé claro, directo y preciso."""
        ),
        ("human", "CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"),
    ])

    def format_docs(items: List[Document]) -> str:
        return "\n\n---\n\n".join(
            f"[{item.metadata.get('title', 'Sin título')}]\n{item.page_content}"
            for item in items
        )

    setup_and_retrieval = RunnableParallel(
        question=RunnablePassthrough(),
        documents=retriever,
    )
    answer_from_documents = (
        {
            "context": lambda item: format_docs(item["documents"]),
            "question": lambda item: item["question"],
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    pipeline = setup_and_retrieval.assign(answer=answer_from_documents)
    return pipeline, chunks


print("Construyendo el índice...")
rag_pipeline, indexed_chunks = build_pipeline(company_documents)
titles = [document["title"] for document in company_documents]
print(f"✅ Pipeline listo: {len(titles)} documentos, {len(indexed_chunks)} chunks")


Construyendo el índice...


/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/llama-nemotron-embed-vl-1b-v2 in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ Pipeline listo: 8 documentos, 11 chunks


## 5. Crear la interfaz Gradio

Esta celda define `demo` y la función `chat`, que consulta el pipeline y muestra la respuesta junto con las fuentes recuperadas.

In [6]:
# Crear la interfaz de Gradio usando el pipeline construido en la sección 4.
# No reconstruimos el índice aquí: reutilizamos rag_pipeline.

def chat(question: str):
    if not question or not question.strip():
        return "Escribe una pregunta para comenzar.", ""

    try:
        result = rag_pipeline.invoke(question.strip())
        answer = result.get("answer", "No se generó una respuesta.")
        retrieved_docs = result.get("documents", [])

        if retrieved_docs:
            sources = "**Fuentes recuperadas:**\\n\\n" + "\\n\\n".join(
                f"- **{doc.metadata.get('title', 'Sin título')}**"
                for doc in retrieved_docs
            )
        else:
            sources = (
                "**Fuentes recuperadas:** No se encontraron documentos "
                "por encima del umbral de similitud configurado."
            )

        return answer, sources

    except Exception as e:
        return (
            f"Error al consultar el pipeline: {type(e).__name__}: {e}",
            ""
        )


with gr.Blocks(title="NovaTech — Asistente RAG") as demo:
    gr.Markdown(
        "# Asistente RAG de NovaTech\\n"
        "Haz una pregunta sobre las políticas y procedimientos de los documentos."
    )

    question = gr.Textbox(
        label="Tu pregunta",
        placeholder="Ej.: ¿Quién debe aprobar un descuento comercial del 15 %?",
        lines=2,
    )
    submit = gr.Button("Consultar", variant="primary")

    gr.Markdown("### Respuesta")
    answer_output = gr.Markdown()

    gr.Markdown("### Fuentes recuperadas")
    sources_output = gr.Markdown()

    submit.click(
        fn=chat,
        inputs=question,
        outputs=[answer_output, sources_output],
    )
    question.submit(
        fn=chat,
        inputs=question,
        outputs=[answer_output, sources_output],
    )

print("✅ Interfaz Gradio creada correctamente.")

✅ Interfaz Gradio creada correctamente.


## 6. Ejecutar la aplicación en Colab

Ejecuta esta celda después de las secciones anteriores. `share=True` crea una URL pública temporal. Mantén activa la sesión de Colab mientras uses la aplicación.

In [ ]:
demo.queue().launch(
    share=True,
    debug=True,
    show_error=True,
    prevent_thread_lock=True,
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8b8e192ce2e5c6b37c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Detener o volver a ejecutar

- Para detener el servidor, interrumpe la celda de lanzamiento.
- Si vas a reiniciar Gradio, detén el servidor anterior antes de volver a lanzarlo.
- No ejecutes `gr.close_all()` después de lanzar la aplicación, porque cerrará la interfaz.
- El enlace público cambia cada vez que se inicia la aplicación.

In [ ]:
# Ejecuta manualmente esta celda solo si necesitas cerrar servidores Gradio anteriores.
# gr.close_all()